# Case study: Alzheimer's disease in AlzKG

Mirrors `../../PrimeKG/case_study/autism.ipynb`'s pattern: search for the disease
across the source vocabularies, then walk the built graph (`ad_kg.csv`) outward from
the Alzheimer's disease node to known drugs, genes, and variants, as a sanity
check that `../knowledge_graph/build_ad_kg.ipynb` produced a sensible graph.

Edges are stored with a direction (`x -> y`), but most queries here don't care about
it, so they look at both sides via the `neighbors()` helper.

In [1]:
import pandas as pd
from IPython.display import display

primekg_data = '../../PrimeKG/datasets/data/'
ad_data = '../datasets/data/'

ad_kg = pd.read_csv(ad_data + 'kg/ad_kg.csv', low_memory=False)
nodes = pd.read_csv(ad_data + 'kg/ad_kg_nodes.csv', low_memory=False)
print(ad_kg.shape, nodes.shape)

def neighbors(node_index, kg=ad_kg):
    # All edges touching node_index, re-oriented so the node is always the 'source' side.
    swap = lambda me, other: (lambda c: 'src_' + c[2:] if c.startswith(me) else
                              'nbr_' + c[2:] if c.startswith(other) else c)
    out = kg[kg['x_index'] == node_index].rename(columns=swap('x_', 'y_'))
    inc = kg[kg['y_index'] == node_index].rename(columns=swap('y_', 'x_'))
    return pd.concat([out, inc], ignore_index=True)

(27537, 13) (15813, 5)


## Search for "alzheimer" across the source vocabularies

In [2]:
dfs = [
    ('mondo (PrimeKG)', 'name', primekg_data + 'mondo/mondo_terms.csv'),
    ('GWAS Catalog', 'y_name', ad_data + 'gwas_catalog/gwas_gene_disease.csv'),
    ('ClinVar', 'y_name', ad_data + 'clinvar/clinvar_ad_variant_disease.csv'),
    ('ClinicalTrials.gov', 'y_name', ad_data + 'aact/clinicaltrials_drug_disease.csv'),
    ('PubTator3', 'y_name', ad_data + 'literature/literature_triples.csv'),
    ('AlzKG nodes', 'node_name', ad_data + 'kg/ad_kg_nodes.csv'),
]
for name, col, pth in dfs:
    try:
        df = pd.read_csv(pth, low_memory=False)
    except FileNotFoundError:
        print(f'{name}: file not found ({pth}), skipping')
        continue
    x = df[[col]].dropna()
    x = x[x[col].str.contains('alzheimer', case=False)].drop_duplicates()
    print(f'{name}: {x.shape[0]} distinct "alzheimer" names')
display(x.head(10))

mondo (PrimeKG): file not found (../../PrimeKG/datasets/data/mondo/mondo_terms.csv), skipping
GWAS Catalog: 7 distinct "alzheimer" names
ClinVar: 18 distinct "alzheimer" names
ClinicalTrials.gov: 193 distinct "alzheimer" names
PubTator3: 2 distinct "alzheimer" names
AlzKG nodes: 186 distinct "alzheimer" names


,node_name
10393,Alzheimer's disease
10786,Multimodal intervention to Reduce Unwanted Lon...
10861,Alzheimer's Disease biomarkers
10965,Best medical treatment for Alzheimer's disease
10968,Augmented reality technological assistance for...
11273,Omental transposition for mild Alzheimers disease
11284,Alzheimer's Disease Risk Disclosure
11285,Amyloid Brain Imaging and Alzheimer's Disease ...
11298,Alzheimer Disease Education
11332,low dose whole brain radiation to treat Alzhei...


## Alzheimer's disease in AlzKG

In [3]:
AD_ID = 'MONDO:0004975'
ad = nodes[nodes['node_id'] == AD_ID].iloc[0]
ad_nbrs = neighbors(ad['node_index'])
print(f"{ad['node_name']} ({AD_ID}) has {len(ad_nbrs):,} edges to "
      f"{ad_nbrs['nbr_index'].nunique():,} distinct neighbors")
ad_nbrs.groupby(['nbr_type', 'relation']).size().sort_values(ascending=False).to_frame('edges')

Alzheimer's disease (MONDO:0004975) has 12,932 edges to 12,817 distinct neighbors


edges
nbr_type               relation                              
variant                variant_disease                   5543
gene/protein           gene_disease                      3571
clinical_intervention  drug_disease                      1934
drug/chemical/compound drug_disease                      1287
gene/protein           literature_association             253
disease                disease_disease                    150
drug/chemical/compound literature_negative_correlation     73
                       literature_association              56
                       literature_positive_correlation     32
variant                literature_association              11
gene/protein           literature_positive_correlation      8
                       literature_negative_correlation      8
variant                literature_positive_correlation      4
effect/phenotype       phenotype_disease                    2

In [4]:
# Known approved/late-stage AD drugs (cholinesterase inhibitors, NMDA antagonist,
# anti-amyloid antibodies), linked via ClinicalTrials.gov, PharmGKB or PubTator3.
AD_DRUGS = ['donepezil', 'rivastigmine', 'galantamine', 'memantine',
            'lecanemab', 'aducanumab', 'donanemab']
drugs = ad_nbrs[ad_nbrs['nbr_name'].str.lower().isin(AD_DRUGS)]
drugs.groupby('nbr_name')['edge_source'].agg(['count', lambda s: ', '.join(sorted(set(s)))]) \
     .set_axis(['edges', 'sources'], axis=1).sort_values('edges', ascending=False)

,edges,sources
nbr_name,,
aducanumab,2,"ClinicalTrials.gov, PubTator3"
donepezil,2,"ClinicalTrials.gov, PubTator3"
galantamine,2,"ClinicalTrials.gov, PubTator3"
memantine,2,"ClinicalTrials.gov, PubTator3"
lecanemab,2,"ClinicalTrials.gov, PubTator3"
rivastigmine,2,"ClinicalTrials.gov, PubTator3"
donanemab,1,ClinicalTrials.gov


In [5]:
# Known AD risk/causal genes (APOE = strongest late-onset risk gene;
# APP/PSEN1/PSEN2 = early-onset familial AD genes, reached via their ClinVar
# variants and AD subtype nodes rather than a direct gene-AD edge).
AD_GENES = ['APOE', 'APP', 'PSEN1', 'PSEN2', 'TREM2', 'CLU', 'BIN1', 'ABCA7', 'SORL1', 'CR1']
gene_nodes = nodes[(nodes['node_type'] == 'gene/protein') & nodes['node_name'].isin(AD_GENES)]
rows = []
for g in gene_nodes.itertuples():
    nb = neighbors(g.node_index)
    rows.append({
        'gene': g.node_name, 'node_id': g.node_id,
        'direct AD edges': int((nb['nbr_index'] == ad['node_index']).sum()),
        'AD variants located in gene': int(((nb['relation'] == 'variant_gene')).sum()),
        'sources': ', '.join(sorted(set(nb['edge_source']))),
    })
pd.DataFrame(rows).set_index('gene').loc[[g for g in AD_GENES if g in set(gene_nodes['node_name'])]]

,node_id,direct AD edges,AD variants located in gene,sources
gene,,,,
APOE,NCBIGene:348,4,13,"ClinVar, GWAS_Catalog, PharmGKB, PubTator3"
APP,NCBIGene:351,3,230,"ClinVar, GWAS_Catalog, PubTator3"
PSEN1,NCBIGene:5663,1,311,"ClinVar, PubTator3"
PSEN2,NCBIGene:5664,1,141,"ClinVar, PubTator3"
TREM2,NCBIGene:54209,2,0,"GWAS_Catalog, PubTator3"
CLU,NCBIGene:1191,2,0,"GWAS_Catalog, PubTator3"
BIN1,NCBIGene:274,2,0,"GWAS_Catalog, PubTator3"
ABCA7,NCBIGene:10347,2,34,"ClinVar, GWAS_Catalog, PubTator3"
SORL1,NCBIGene:6653,2,1,"ClinVar, GWAS_Catalog, PubTator3"


In [6]:
# Two-hop: drugs connected to a gene that is itself one hop from AD (candidate
# repurposing leads), mirroring the Risperidone-via-target lookup in autism.ipynb.
ad_gene_idx = set(ad_nbrs.loc[ad_nbrs['nbr_type'] == 'gene/protein', 'nbr_index'])
gene_drug = ad_kg[(ad_kg['x_index'].isin(ad_gene_idx) & (ad_kg['y_type'] == 'drug/chemical/compound'))
                  | (ad_kg['y_index'].isin(ad_gene_idx) & (ad_kg['x_type'] == 'drug/chemical/compound'))]
gene_drug = gene_drug.assign(
    gene=gene_drug['x_name'].where(gene_drug['x_type'] == 'gene/protein', gene_drug['y_name']),
    drug=gene_drug['y_name'].where(gene_drug['x_type'] == 'gene/protein', gene_drug['x_name']))
print(f'{gene_drug["drug"].nunique()} drugs are 2 hops from AD through {gene_drug["gene"].nunique()} AD genes')
gene_drug.groupby('gene')['drug'].agg(['nunique', lambda s: ', '.join(sorted(set(s))[:8])]) \
         .set_axis(['n_drugs', 'examples'], axis=1).sort_values('n_drugs', ascending=False).head(10)

329 drugs are 2 hops from AD through 52 AD genes


,n_drugs,examples
gene,,
CYP2D6,228,"2-hydroxymethyl olanzapine, 3,4-methylenedioxy..."
HTR2A,45,"Analgesics, Antiinflammatory agents, non-stero..."
ACE,37,"Ace Inhibitors, Plain, Beta Blocking Agents, C..."
APOE,24,"Antivirals for treatment of HIV infections, co..."
APP,14,"Aluminum Chloride, Glucose, Iron, Lipids, Lipo..."
CHRNA7,10,"donepezil, eculizumab, efgartigimod alfa, gala..."
BCHE,10,"SN-38, chloroprocaine, cocaine, donepezil, iri..."
App,9,"Bendroflumethiazide, Carrageenan, Etodolac, Eu..."
MAPT,6,"Aluminum Chloride, Glucose, Iron, Lipopolysacc..."


In [7]:
# A concrete path: APOE variant rs429358 (the APOE e4-defining SNP) -> AD.
rs = nodes[nodes['node_id'] == 'rs429358']
if len(rs):
    display(neighbors(rs.iloc[0]['node_index'])[['relation', 'display_relation', 'nbr_name', 'nbr_type', 'edge_source']])

,relation,display_relation,nbr_name,nbr_type,edge_source
0,variant_disease,conflicting classifications of pathogenicity; ...,Alzheimer disease 2,disease,ClinVar
1,variant_disease,conflicting classifications of pathogenicity; ...,Primary degenerative dementia of the Alzheimer...,disease,ClinVar
2,variant_disease,conflicting classifications of pathogenicity; ...,Alzheimer's disease,disease,ClinVar
3,variant_disease,conflicting classifications of pathogenicity; ...,Alzheimer disease 4,disease,ClinVar
4,variant_gene,located in,APOE,gene/protein,ClinVar
5,variant_disease,associated with,Alzheimer's disease,disease,GWAS_Catalog
6,variant_disease,associated with,Alzheimer's disease biomarker measurement,effect/phenotype,GWAS_Catalog
7,variant_disease,associated with,family history of Alzheimer's disease,disease,GWAS_Catalog
8,variant_disease,associated with,late-onset Alzheimers disease,disease,GWAS_Catalog
9,variant_disease,associated,Alzheimer's disease,disease,PharmGKB
